<a href="https://www.kaggle.com/code/riteshkumarweb/logistic-softmax-regression-multinomial-regression?scriptVersionId=314823975" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/uciml/iris/Iris.csv
/kaggle/input/datasets/organizations/uciml/iris/database.sqlite


In [ ]:
# 📊 Softmax Regression / Multinomial Logistic Regression

# Both are mostly the same thing

# Used for Multi-class Classification
# → When target has more than 2 classes

# Example:
# Class 0 = Cat 🐱
# Class 1 = Dog 🐶
# Class 2 = Bird 🐦

# Normal Logistic Regression
# → Used for only 2 classes (Binary Classification)

# Softmax Regression
# → Used for more than 2 classes

# 🎯 It predicts probability for each class

# Example output:
# [0.10, 0.70, 0.20]

# Meaning:
# Cat = 10%
# Dog = 70%
# Bird = 20%

# Final prediction:
# Highest probability wins → Dog 🐶

In [5]:
df = pd.read_csv('/kaggle/input/datasets/organizations/uciml/iris/Iris.csv')

In [6]:
df

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


In [7]:
df.sample(10)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
91,92,6.1,3.0,4.6,1.4,Iris-versicolor
93,94,5.0,2.3,3.3,1.0,Iris-versicolor
136,137,6.3,3.4,5.6,2.4,Iris-virginica
66,67,5.6,3.0,4.5,1.5,Iris-versicolor
67,68,5.8,2.7,4.1,1.0,Iris-versicolor
46,47,5.1,3.8,1.6,0.2,Iris-setosa
35,36,5.0,3.2,1.2,0.2,Iris-setosa
132,133,6.4,2.8,5.6,2.2,Iris-virginica
105,106,7.6,3.0,6.6,2.1,Iris-virginica
65,66,6.7,3.1,4.4,1.4,Iris-versicolor


In [8]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [9]:
df['Species'].value_counts()

Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

In [14]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['Species'] = encoder.fit_transform(df['Species'])

In [15]:
df.sample(10)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
24,25,4.8,3.4,1.9,0.2,0
118,119,7.7,2.6,6.9,2.3,2
109,110,7.2,3.6,6.1,2.5,2
25,26,5.0,3.0,1.6,0.2,0
7,8,5.0,3.4,1.5,0.2,0
12,13,4.8,3.0,1.4,0.1,0
78,79,6.0,2.9,4.5,1.5,1
83,84,6.0,2.7,5.1,1.6,1
91,92,6.1,3.0,4.6,1.4,1
59,60,5.2,2.7,3.9,1.4,1


In [17]:
df.columns.tolist()

['Id',
 'SepalLengthCm',
 'SepalWidthCm',
 'PetalLengthCm',
 'PetalWidthCm',
 'Species']

In [19]:
df.drop(columns=['Id'], inplace=True)

In [20]:
df

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [24]:
X = df.iloc[:,0:4]
X

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [26]:
y = df.iloc[:,-1]
y

0      0
1      0
2      0
3      0
4      0
      ..
145    2
146    2
147    2
148    2
149    2
Name: Species, Length: 150, dtype: int64

In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test =train_test_split(X,y,test_size=0.2,random_state=42)

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix
# Use 'max_iter' if your data is complex and needs more time to converge
model = LogisticRegression(multi_class='multinomial')#⚠️
# Removed multi_class parameter because sklearn automatically detects
# multiclass classification and uses multinomial (Softmax Regression) by default
# in newer versions, so no need to manually set multi_class='multinomial'
model = LogisticRegression()#in this if needed use max_iter
model.fit(X_train,y_train)
y_pred = model.predict(X_test)
score = (accuracy_score(y_test,y_pred))
print(f'The score of this model is {score*100}% of this dataset ')

The score of this model is 100.0% of this dataset 


In [37]:
confusion_matrix(y_test,y_pred)

array([[10,  0,  0],
       [ 0,  9,  0],
       [ 0,  0, 11]])